# Vietnamese Emotion Classification - Advanced CafeBERT Fine-tuning (with Underthesea Tokenizer)

**🌟 Google Colab Ready!** This notebook is optimized for Google Colab with GPU acceleration (20GB RAM recommended).

## 📝 Two Ways to Use This Notebook:

### 🚀 Option 1: Skip Training (Quick Start)
**If you already have a trained model:**
1. Mount Google Drive
2. Set `SKIP_TRAINING = True` in the Quick Start cell
3. Jump to prediction sections

### 🎓 Option 2: Train from Scratch
**For training a new model:**
1. **Upload training data** to Google Drive at: `MyDrive/thesis/data/`
   - Required files: `train_nor_811.xlsx`, `valid_nor_811.xlsx`, `test_nor_811.xlsx`

2. **Enable GPU** in Colab: Runtime → Change runtime type → GPU
   - **A100 (40GB)** - Best performance, batch size 32
   - **V100 (16GB)** - Great performance, batch size 16 (default)
   - **T4 (16GB)** - Good for free tier, reduce batch size to 8 if needed
   - **TPU v2-8** - Also supported but GPU recommended

3. **Run cells in order** - the notebook will:
   - Mount your Google Drive
   - Detect and configure GPU automatically
   - Load training data from Drive
   - Train the model using GPU with mixed precision (FP16)
   - Save the model back to Drive
   - Run predictions with your trained model

---

This notebook implements **state-of-the-art** fine-tuning techniques for BERT-based emotion classification:

## 🚀 Advanced Techniques Implemented:

### 1. **Layer-wise Learning Rate Decay (LLRD)**
- Applies different learning rates to each layer
- Base LR for top layer: **3.5e-6**
- Decay factor: **0.95**
- Classifier head gets **10x** higher LR
- **Preserves pre-trained knowledge** in lower layers

### 2. **Optimized Batch Size & Gradient Accumulation**
- Effective batch size: **16-32** (optimal for BERT)
- Gradient accumulation to simulate larger batches
- **Scheduler steps after EVERY batch** (critical!)
- Stable training with limited memory

### 3. **Early Stopping & Best Model Selection**
- Monitors validation F1 (weighted)
- Patience: 2 epochs
- Saves best model automatically

---

## 📊 Expected Results

**Your Dataset (NEU-ESC):**
- 7 emotions (Enjoyment, Disgust, Other, Sadness, Anger, Fear, Surprise)

**Expected Results:**

| Metric | Expected Range |
|--------|----------------|
| Accuracy | **70-80%** |
| F1 Macro | **50-65%** |
| F1 Weighted | **68-78%** |

---

**Let's get started!**


## 🔧 Google Colab Setup - Mount Google Drive

**Important:** Run this cell first to access your data on Google Drive


In [ ]:
# Mount Google Drive to access training data
from google.colab import drive
drive.mount('/content/drive')

print("✓ Google Drive mounted successfully")
print("✓ Your data should be in: /content/drive/MyDrive/thesis/data")

# Check for TPU and set up if available
import os
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    TPU_AVAILABLE = True
    print("✓ TPU detected and configured")
except ImportError:
    TPU_AVAILABLE = False
    print("ℹ️  TPU not available, will use GPU/CPU")


## 🚀 Quick Start: Load Pre-trained Model (Skip Training)

**Run this section if you already have a trained model and want to skip training.**

Set `SKIP_TRAINING = True` below to load your saved model and jump to predictions.


In [ ]:
# ============================================================================
# QUICK START: Load existing model and skip training
# ============================================================================

SKIP_TRAINING = False  # Set to True to load saved model and skip training

if not SKIP_TRAINING:
    print("=" * 80)
    print("📚 TRAINING MODE")
    print("=" * 80)
    print("Continue running cells below to train a new model.")
    print("Skip this Quick Start section and proceed to Configuration.")
    print("=" * 80)

if SKIP_TRAINING:
    print("🚀 Loading pre-trained model from Google Drive...\n")

    # Install required packages
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'transformers', 'datasets', 'torch',
                   'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'openpyxl'],
                   check=False)

    import os
    import torch
    import json
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    # Model path
    MODEL_PATH = '/content/drive/MyDrive/thesis/emotion_classifier_cafebert'

    # Load model and tokenizer
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

    # Load label mappings
    with open(os.path.join(MODEL_PATH, 'label_mappings.json'), 'r') as f:
        label_maps = json.load(f)
        label2id = label_maps['label2id']
        id2label = {int(k): v for k, v in label_maps['id2label'].items()}
        num_classes = len(id2label)

    # Set device (no TPU for inference after restart)
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print(f"✓ Using device: cuda")
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device('cpu')
        print(f"✓ Using device: cpu")

    model = model.to(device)
    USE_TPU = False

    print(f"✓ Model loaded from: {MODEL_PATH}")
    print(f"✓ Number of classes: {num_classes}")
    print(f"✓ Emotions: {list(id2label.values())}")
    print(f"\n✅ Ready for predictions! Jump to cell 'Predict on HuggingFace Dataset'")
    print("=" * 80)


## 📋 Configuration Section

**Adjust these hyperparameters based on your needs:**


In [ ]:
# ============================================================================
# CONFIGURATION - Adjust these hyperparameters as needed
# ============================================================================

# Random seeds for multiple runs (to measure average performance)
# Set NUM_RUNS to control how many seeds to use (1-5)
NUM_RUNS = 5  # Change this to run more/fewer experiments
SEEDS = [42, 123, 456, 789, 1024][:NUM_RUNS]

# Data paths - Using Google Drive (make sure your data is uploaded to this location)
DATA_DIR = '/content/drive/MyDrive/thesis/data'
MODEL_SAVE_PATH = '/content/drive/MyDrive/thesis/emotion_classifier_cafebert'

# Model selection
MODEL_NAME = 'uitnlp/CafeBERT'  # CafeBERT model for Vietnamese

# Batch size settings optimized for 20GB GPU RAM
# Note: Large model requires more memory - reduce batch size if OOM errors occur
PER_DEVICE_TRAIN_BATCH_SIZE = 16  # If you have enough GPU memory
PER_DEVICE_EVAL_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 4   # Effective batch size = 16 * 2 = 32

# Layer-wise Learning Rate Decay (LLRD) settings
BASE_LEARNING_RATE = 5e-5 # Base learning rate
LR_DECAY_FACTOR = 0.95  # Decay factor (0.65-0.95 recommended)
CLASSIFIER_LR_MULTIPLIER = 10.0  # Classifier head gets higher LR

# Training settings
NUM_EPOCHS = 20
WARMUP_RATIO = 0.1  # 10% of training steps for warmup
WEIGHT_DECAY = 0.01
EARLY_STOPPING_PATIENCE = 5

# Data augmentation
USE_OVERSAMPLING = False  # Set to True to apply random oversampling

# Text preprocessing options
USE_UNDERTHESEA_TOKENIZER = True  # Set to True to use underthesea word segmentation

print("✓ Configuration loaded")
print(f"  Model: {MODEL_NAME}")
print(f"  Data dir: {DATA_DIR}")
print(f"  Model save path: {MODEL_SAVE_PATH}")
print(f"  Batch size (train): {PER_DEVICE_TRAIN_BATCH_SIZE}")
print(f"  Batch size (eval): {PER_DEVICE_EVAL_BATCH_SIZE}")
print(f"  Effective batch size: {PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  Base LR: {BASE_LEARNING_RATE:.2e}")
print(f"  Oversampling: {USE_OVERSAMPLING}")
print(f"  Underthesea tokenizer: {USE_UNDERTHESEA_TOKENIZER}")
print(f"  Number of runs: {NUM_RUNS} (seeds: {SEEDS})")
print(f"\n💡 GPU Memory Tips:")
print(f"  - If OOM error: reduce PER_DEVICE_TRAIN_BATCH_SIZE to 8 or 4")
print(f"  - With 20GB GPU: batch size 16 should work well")
print(f"  - FP16 mixed precision will be enabled automatically on GPU")


## 1. Install & Import Libraries


In [ ]:
%pip install -q transformers datasets torch scikit-learn pandas numpy matplotlib seaborn openpyxl underthesea wandb


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                              f1_score, precision_recall_fscore_support)
from sklearn.utils import resample
import torch
import torch.nn as nn
from torch.optim import AdamW
import json
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    get_linear_schedule_with_warmup
)
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

# Check device - prioritize CUDA GPU, then CPU
# Note: Random seeds are set per-run in the training loop for multi-seed experiments
USE_TPU = False

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✓ Using device: {device}")
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
    print(f"✓ Using device: {device} (Metal Performance Shaders)")
else:
    device = torch.device('cpu')
    print(f"✓ Using device: {device}")
    print("  ⚠️  Warning: No GPU detected. Training will be slower.")

print("✓ All libraries imported successfully")


## 📊 Weights & Biases (wandb) Setup

**Track your experiments with wandb for better visualization and comparison**

In [ ]:
import wandb
from google.colab import userdata

# Automatically retrieve wandb API key from Colab secrets
try:
    wandb_api_key = userdata.get('WANDB_API_KEY')
    wandb.login(key=wandb_api_key)
    print("✓ Successfully logged into wandb using Colab secrets")
except Exception as e:
    print(f"⚠️  Could not retrieve WANDB_API_KEY from Colab secrets: {e}")
    print("   Please add your wandb API key to Colab secrets:")
    print("   1. Click the 🔑 icon in the left sidebar")
    print("   2. Add a new secret with name: WANDB_API_KEY")
    print("   3. Paste your API key from https://wandb.ai/authorize")
    print("\n   Or login manually:")
    wandb.login()

# Initialize wandb project
wandb.init(
    project="emotion-classification-baseline",
    name=f"cafebert-underthesea-baseline",
    config={
        "model": "uitnlp/CafeBERT",
        "tokenizer": "underthesea",
        "experiment_type": "baseline"
    }
)
print("✓ wandb initialized")

## 3. Layer-wise Learning Rate Decay (LLRD) Implementation

**Key Concept:** Different layers get different learning rates
- **Classifier head:** Highest LR (3.5e-5 = 10× base)
- **Top encoder layers:** Base LR (3.5e-6)  
- **Middle layers:** Gradually decreasing  
- **Embeddings:** Lowest LR (~1.9e-6)

**Formula:** `LR_layer_i = LR_base × (decay_factor)^(num_layers - i)`

**Why?** Lower layers encode general linguistic knowledge → preserve it with lower LR


In [ ]:
def get_optimizer_grouped_parameters(model, base_lr=5e-5, lr_decay_factor=0.95,
                                    weight_decay=0.01, classifier_lr_multiplier=10.0):
    """Create parameter groups with layer-wise learning rate decay."""

    no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
    optimizer_grouped_parameters = []

    # Get model type
    if hasattr(model, 'roberta'):
        encoder = model.roberta
        model_type = 'roberta'
    elif hasattr(model, 'bert'):
        encoder = model.bert
        model_type = 'bert'
    else:
        raise ValueError("Model type not supported for LLRD")

    num_layers = len(encoder.encoder.layer)

    print(f"\n{'='*80}")
    print("LAYER-WISE LEARNING RATE DECAY (LLRD)")
    print(f"{'='*80}")
    print(f"Model: {model_type} | Layers: {num_layers} | Base LR: {base_lr:.2e} | Decay: {lr_decay_factor}")
    print("\nLearning rates by layer:")

    # 1. Classifier head (highest LR)
    classifier_lr = base_lr * classifier_lr_multiplier
    optimizer_grouped_parameters.extend([
        {'params': [p for n, p in model.classifier.named_parameters() if not any(nd in n for nd in no_decay)],
         'lr': classifier_lr, 'weight_decay': weight_decay},
        {'params': [p for n, p in model.classifier.named_parameters() if any(nd in n for nd in no_decay)],
         'lr': classifier_lr, 'weight_decay': 0.0}
    ])
    print(f"  Classifier head: {classifier_lr:.2e}")

    # 2. Encoder layers (top to bottom with decay)
    for layer_idx in range(num_layers - 1, -1, -1):
        layer = encoder.encoder.layer[layer_idx]
        lr = base_lr * (lr_decay_factor ** (num_layers - 1 - layer_idx))

        optimizer_grouped_parameters.extend([
            {'params': [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
             'lr': lr, 'weight_decay': weight_decay},
            {'params': [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)],
             'lr': lr, 'weight_decay': 0.0}
        ])

        if layer_idx % 3 == 0:
            print(f"  Layer {layer_idx}: {lr:.2e}")

    # 3. Embeddings (lowest LR)
    embedding_lr = base_lr * (lr_decay_factor ** num_layers)
    optimizer_grouped_parameters.extend([
        {'params': [p for n, p in encoder.embeddings.named_parameters() if not any(nd in n for nd in no_decay)],
         'lr': embedding_lr, 'weight_decay': weight_decay},
        {'params': [p for n, p in encoder.embeddings.named_parameters() if any(nd in n for nd in no_decay)],
         'lr': embedding_lr, 'weight_decay': 0.0}
    ])
    print(f"  Embeddings: {embedding_lr:.2e}")
    print(f"{'='*80}\n")

    return optimizer_grouped_parameters

print("✓ LLRD function defined")


In [ ]:
class AdvancedTrainer(Trainer):
    """Advanced Trainer with LLRD Support."""

    def __init__(self, *args, llrd_config=None, **kwargs):
        self.llrd_config = llrd_config or {}
        super().__init__(*args, **kwargs)

    def create_optimizer(self):
        """Create optimizer with LLRD parameter groups."""
        if self.optimizer is None:
            optimizer_grouped_parameters = get_optimizer_grouped_parameters(
                self.model,
                base_lr=self.llrd_config.get('base_lr', 1.5e-6),
                lr_decay_factor=self.llrd_config.get('lr_decay_factor', 0.95),
                weight_decay=self.args.weight_decay,
                classifier_lr_multiplier=self.llrd_config.get('classifier_lr_multiplier', 10.0)
            )

            self.optimizer = AdamW(optimizer_grouped_parameters, betas=(0.9, 0.999), eps=1e-8)

        return self.optimizer

    def create_scheduler(self, num_training_steps: int, optimizer=None):
        """Create LR scheduler. CRITICAL: Steps after every batch!"""
        if self.lr_scheduler is None:
            warmup_steps = int(num_training_steps * self.args.warmup_ratio)

            self.lr_scheduler = get_linear_schedule_with_warmup(
                optimizer=self.optimizer if optimizer is None else optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=num_training_steps
            )

            print(f"\n✓ Scheduler created: {num_training_steps} steps, {warmup_steps} warmup")
            print("  ⚠️  CRITICAL: Scheduler steps after EVERY batch\n")

        return self.lr_scheduler

print("✓ AdvancedTrainer class defined")


## 4. Load and Prepare Data

Loading NEU-ESC dataset (pre-processed in DataValidationAndPreprocess.ipynb) and applying underthesea word segmentation


In [ ]:
# Underthesea word segmentation for Vietnamese
_underthesea_available = False
if USE_UNDERTHESEA_TOKENIZER:
    try:
        from underthesea import word_tokenize as uts_word_tokenize
        _underthesea_available = True
        print("✓ Underthesea word tokenizer loaded")
    except ImportError:
        print("⚠️  Underthesea not available, falling back to simple tokenization")

def tokenize_vietnamese(text):
    """Tokenize Vietnamese text using underthesea word segmentation.
    
    Vietnamese words can be multi-syllable (e.g., 'học sinh' = student).
    Underthesea segments text into proper Vietnamese words.
    """
    if text is None:
        return ""
    
    t = str(text).strip()
    
    # Apply underthesea word segmentation if available
    if _underthesea_available:
        t = uts_word_tokenize(t, format='text')  # Returns text with underscores for compound words
    
    return t

print(f"✓ Vietnamese text preprocessing ready")
if USE_UNDERTHESEA_TOKENIZER and _underthesea_available:
    print(f"✓ Underthesea word segmentation enabled")


In [ ]:
# Create model save directory if it doesn't exist
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
print(f"✓ Model save directory ready: {MODEL_SAVE_PATH}")

# Load pre-processed datasets
train_df = pd.read_csv(os.path.join(DATA_DIR, 'processed/train_processed.csv'))
val_df = pd.read_csv(os.path.join(DATA_DIR, 'processed/val_processed.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'processed/test_processed.csv'))

print(f"\n✓ Datasets loaded from Google Drive")
print(f"  Train: {train_df.shape[0]} samples")
print(f"  Val: {val_df.shape[0]} samples")
print(f"  Test: {test_df.shape[0]} samples")

print(train_df.columns)
print(val_df.columns)
print(test_df.columns)

train_df = train_df.drop(["Emotion", "text_length","word_count"], axis=1)
val_df = val_df.drop(["Emotion"],axis = 1)
test_df = test_df.drop(["Emotion"],axis=1)

train_df.columns = ['text', 'label']
val_df.columns = ['text', 'label']
test_df.columns = ['text', 'label']

# Basic data cleaning and underthesea tokenization
for df in [train_df, val_df, test_df]:
    df.dropna(subset=['text', 'label'], inplace=True)
    df['text'] = df['text'].astype(str).str.strip()
    df['label'] = df['label'].astype(str).str.strip()
    
    # Apply underthesea tokenization
    if USE_UNDERTHESEA_TOKENIZER:
        df['text'] = df['text'].apply(tokenize_vietnamese)
        print(f"✓ Applied underthesea tokenization to {df.shape[0]} samples")

# Show samples
print(f"\n{'='*80}")
print("SAMPLE TEXT AFTER TOKENIZATION")
print(f"{'='*80}")
print(f"Sample 1: {train_df['text'].iloc[0][:150]}...")
print(f"Sample 2: {train_df['text'].iloc[1][:150]}...")
print(f"Sample 3: {train_df['text'].iloc[2][:150]}...")
print(f"{'='*80}")

print("\n✓ Data loaded and tokenized successfully")


## 5. Analyze Class Distribution

Understanding the class distribution in the training data


In [ ]:
# Analyze distribution
label_counts = train_df['label'].value_counts()
print("Emotion distribution (Training set):")
print(label_counts)
print(f"\nImbalance ratio: {label_counts.max() / label_counts.min():.1f}:1")

# Visualize
plt.figure(figsize=(12, 6))
label_counts.sort_index().plot(kind='bar', color='steelblue', edgecolor='black')
plt.xlabel('Emotion', fontsize=12, fontweight='bold')
plt.ylabel('Count', fontsize=12, fontweight='bold')
plt.title('Training Set - Emotion Distribution', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Create label mappings
unique_labels = sorted(train_df['label'].unique())
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}
num_classes = len(unique_labels)

# Add numeric labels
train_df['labels'] = train_df['label'].map(label2id)
val_df['labels'] = val_df['label'].map(label2id)
test_df['labels'] = test_df['label'].map(label2id)

print(f"\n✓ Label mappings created ({num_classes} classes)")


## 6. Tokenize Data & Load Model


In [ ]:
# Convert to Dataset format
train_dataset = Dataset.from_pandas(train_df[['text', 'labels']].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[['text', 'labels']].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[['text', 'labels']].reset_index(drop=True))

print(f"✓ Datasets converted to HuggingFace format")

# Load model and tokenizer
print(f"\nLoading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id
)

print(f"✓ Model and tokenizer loaded")
print(f"✓ Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

# Tokenize
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=256)

train_tokenized = train_dataset.map(tokenize_function, batched=True)
val_tokenized = val_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

print(f"\n✓ Tokenized {len(train_tokenized)} training samples")
print(f"✓ Tokenized {len(val_tokenized)} validation samples")
print(f"✓ Tokenized {len(test_tokenized)} test samples")


## 7. Train Advanced Model with LLRD

Training with:
- ✅ Layer-wise Learning Rate Decay (LLRD)  
- ✅ Gradient accumulation  
- ✅ Early stopping


In [ ]:
# Define metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_weighted = f1_score(labels, predictions, average='weighted')

    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted
    }

# LLRD configuration
llrd_config = {
    'base_lr': BASE_LEARNING_RATE,
    'lr_decay_factor': LR_DECAY_FACTOR,
    'classifier_lr_multiplier': CLASSIFIER_LR_MULTIPLIER
}

# Store results from all runs
all_results = []

print(f"\n{'='*80}")
print(f"MULTI-SEED TRAINING: {len(SEEDS)} runs")
print(f"Seeds: {SEEDS}")
print(f"{'='*80}\n")


In [ ]:
# MULTI-SEED TRAINING LOOP
best_f1_weighted = 0
best_seed = None
best_model_ref = None  # Keep reference to best model in memory

for run_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*80}")
    print(f"🚀 RUN {run_idx + 1}/{len(SEEDS)} - Seed: {seed}")
    print(f"{'='*80}\n")
    
    # Set random seeds for reproducibility
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    
    # Reload model for each run (fresh weights)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_classes,
        id2label=id2label,
        label2id=label2id
    )
    model = model.to(device)
    
    # Training arguments with current seed
    training_args = TrainingArguments(
        output_dir=f'./results_advanced_seed_{seed}',
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=BASE_LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        logging_dir=f'./logs_advanced_seed_{seed}',
        logging_steps=50,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1_weighted',
        greater_is_better=True,
        save_total_limit=2,
        tpu_num_cores=8 if USE_TPU else None,
        fp16=(device.type == 'cuda'),
        dataloader_num_workers=0,
        seed=seed,
    )
    
    # Create trainer
    trainer = AdvancedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
        llrd_config=llrd_config
    )
    
    # Train
    train_result = trainer.train()
    
    # Evaluate on test set
    test_results = trainer.evaluate(test_tokenized)
    
    # Store results
    run_results = {
        'seed': seed,
        'run': run_idx + 1,
        'test_accuracy': test_results['eval_accuracy'],
        'test_f1_macro': test_results['eval_f1_macro'],
        'test_f1_weighted': test_results['eval_f1_weighted'],
        'train_runtime': train_result.metrics['train_runtime'],
    }
    all_results.append(run_results)
    
    print(f"\n✅ Run {run_idx + 1} Complete (Seed {seed}):")
    print(f"   Test Accuracy:   {run_results['test_accuracy']:.4f}")
    print(f"   Test F1 Macro:   {run_results['test_f1_macro']:.4f}")
    print(f"   Test F1 Weighted: {run_results['test_f1_weighted']:.4f}")
    print(f"   Training Time:   {run_results['train_runtime']:.2f}s")
    
    # Track best model - keep the model in memory
    if run_results['test_f1_weighted'] > best_f1_weighted:
        # Delete previous best model to free memory
        if best_model_ref is not None:
            del best_model_ref
            torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
        best_f1_weighted = run_results['test_f1_weighted']
        best_seed = seed
        # Keep reference to the best model (trainer.model has best model due to load_best_model_at_end=True)
        best_model_ref = trainer.model
        print(f"   🏆 New best model! (F1 Weighted: {best_f1_weighted:.4f})")
        
        # Clean up trainer but NOT the model (we kept the reference)
        del trainer
    else:
        # Not the best - clean up both trainer and model
        del trainer
        del model
    
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"\n{'='*80}")
print("✅ ALL TRAINING RUNS COMPLETED!")
print(f"{'='*80}")
print(f"\n🏆 Best model: Seed {best_seed} with F1 Weighted = {best_f1_weighted:.4f}")


In [ ]:
# ============================================================================
# AGGREGATE RESULTS FROM ALL RUNS
# ============================================================================
import matplotlib.pyplot as plt

# Convert results to DataFrame for easy analysis
results_df = pd.DataFrame(all_results)

# Calculate statistics
print(f"\n{'='*80}")
print("📊 AGGREGATED RESULTS ACROSS ALL SEEDS")
print(f"{'='*80}\n")

# Individual run results
print("Individual Run Results:")
print("-" * 70)
print(f"{'Run':<6} {'Seed':<8} {'Accuracy':<12} {'F1 Macro':<12} {'F1 Weighted':<12}")
print("-" * 70)
for _, row in results_df.iterrows():
    print(f"{row['run']:<6} {row['seed']:<8} {row['test_accuracy']:<12.4f} {row['test_f1_macro']:<12.4f} {row['test_f1_weighted']:<12.4f}")
print("-" * 70)

# Summary statistics
print(f"\n{'='*80}")
print("SUMMARY STATISTICS (Mean ± Std)")
print(f"{'='*80}")
metrics = ['test_accuracy', 'test_f1_macro', 'test_f1_weighted']
metric_names = ['Accuracy', 'F1 Macro', 'F1 Weighted']

for metric, name in zip(metrics, metric_names):
    mean_val = results_df[metric].mean()
    std_val = results_df[metric].std()
    min_val = results_df[metric].min()
    max_val = results_df[metric].max()
    print(f"{name:<15}: {mean_val:.4f} ± {std_val:.4f}  (min: {min_val:.4f}, max: {max_val:.4f})")

print(f"\nTotal training time: {results_df['train_runtime'].sum():.2f}s")
print(f"Average time per run: {results_df['train_runtime'].mean():.2f}s")
print(f"{'='*80}\n")

# Plot results comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric, name in zip(axes, metrics, metric_names):
    values = results_df[metric].values
    seeds = results_df['seed'].values
    
    bars = ax.bar(range(len(seeds)), values, color='steelblue', edgecolor='black')
    ax.axhline(y=values.mean(), color='red', linestyle='--', label=f'Mean: {values.mean():.4f}')
    ax.fill_between([-0.5, len(seeds)-0.5], values.mean()-values.std(), values.mean()+values.std(), 
                    alpha=0.2, color='red', label=f'±1 Std: {values.std():.4f}')
    
    ax.set_xlabel('Seed')
    ax.set_ylabel(name)
    ax.set_title(f'{name} Across Seeds')
    ax.set_xticks(range(len(seeds)))
    ax.set_xticklabels(seeds)
    ax.legend(loc='lower right')
    ax.set_ylim(min(values) - 0.05, max(values) + 0.05)

plt.tight_layout()
plt.savefig('multi_seed_results.png', dpi=150, bbox_inches='tight')
plt.show()

# Save results to CSV
results_df.to_csv('multi_seed_results.csv', index=False)
print(f"✓ Results saved to 'multi_seed_results.csv'")

## 8. Save Model to Google Drive

Save the trained model to your Google Drive for later use


In [ ]:
# Create versioned model path with model name and data file name
import datetime
import os
import shutil

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
model_name = "CafeBert_Advanced"

# Hard-coded training data file name
data_file_name = "ORIGINAL"

versioned_model_name = f"{model_name}_{data_file_name}_seed{best_seed}_{timestamp}"
versioned_model_path = os.path.join(os.path.dirname(MODEL_SAVE_PATH), versioned_model_name)

# Save the best model from memory to Google Drive
print(f"💾 Saving BEST model to Google Drive...")
print(f"📝 Model version: {versioned_model_name}")
print(f"📊 Training data: {data_file_name}")
print(f"🏆 Best seed: {best_seed} (F1 Weighted: {best_f1_weighted:.4f})")

# Use the best model reference kept in memory (avoids loading from disk)
best_model_ref.save_pretrained(versioned_model_path)
tokenizer.save_pretrained(versioned_model_path)

# Save label mappings for later use
import json
label_mappings = {
    'label2id': label2id,
    'id2label': id2label
}
with open(os.path.join(versioned_model_path, 'label_mappings.json'), 'w', encoding='utf-8') as f:
    json.dump(label_mappings, f, ensure_ascii=False, indent=2)

# Save model metadata (including multi-seed results)
model_metadata = {
    'model_name': model_name,
    'data_file': data_file_name,
    'timestamp': timestamp,
    'version': versioned_model_name,
    'base_model': MODEL_NAME,
    'training_date': datetime.datetime.now().isoformat(),
    'best_seed': best_seed,
    'best_f1_weighted': best_f1_weighted,
    'all_seeds': SEEDS,
    'all_results': all_results
}
with open(os.path.join(versioned_model_path, 'model_metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(model_metadata, f, ensure_ascii=False, indent=2)

print(f"✓ Model saved successfully to: {versioned_model_path}")
print(f"  - Model weights")
print(f"  - Tokenizer")
print(f"  - Label mappings")
print(f"  - Model metadata")
print(f"\n📁 You can access this versioned model from your Google Drive anytime!")

# Finish wandb run
wandb.finish()ssss
print("✓ wandb run finished")

# Auto timeout to save Colab resources
print("\n" + "="*80)
print("🔌 AUTO TIMEOUT: Disconnecting runtime to save resources...")
print("="*80)
print("Training complete! The runtime will disconnect in 60 seconds.")
print(f"Your versioned model ({versioned_model_name}) is safely saved to Google Drive.")

import time
time.sleep(60)

# Disconnect runtime
from google.colab import runtime
runtime.unassign()
